## 🎯 Learning Objectives
* Understand the core concept and purpose of Supervised Finetuning (SFT) for Large Language Models (LLMs).
* Learn how to prepare data and configure a pre-trained LLM for SFT using Hugging Face Transformers and the `trl` library.
* Implement a practical SFT workflow, including model loading, dataset preparation, and training with `SFTTrainer`.
* Interpret training outputs and understand the performance trade-offs and typical use cases of SFT.


## Supervised Finetuning (SFT) with Hugging Face Transformers

Supervised Finetuning (SFT) is a crucial technique for adapting a pre-trained Large Language Model (LLM) to perform specific tasks or adhere to particular styles, instructions, or domains. Think of a pre-trained LLM as a brilliant, highly knowledgeable generalist student who has read almost every book in the world. While incredibly capable, this student might not be immediately proficient in a very niche skill, like writing legal briefs in a specific jurisdiction or generating creative poetry in the style of a particular poet.

SFT is like giving this generalist student a focused, intensive bootcamp. You provide them with a curated set of examples (input-output pairs) that demonstrate exactly how you want them to perform the new task. For instance, if you want the LLM to summarize medical research papers, you'd provide many examples of medical papers paired with their expert-written summaries. The model then learns from these examples, adjusting its internal parameters to better align with the desired output for that specific task.

### Why SFT?

1.  **Domain Adaptation**: Make a general LLM proficient in a specialized field (e.g., finance, healthcare, coding).
2.  **Instruction Following**: Teach the model to follow complex instructions or specific prompt formats.
3.  **Style Transfer**: Guide the model to generate text in a particular tone, voice, or style.
4.  **Task Specialization**: Improve performance on specific tasks like summarization, translation, question answering, or code generation.

### The SFT Process:

1.  **Start with a Pre-trained LLM**: Begin with a foundational model (e.g., Llama-3, Gemma, Mistral) that has already learned vast amounts of language knowledge from diverse data.
2.  **Prepare a High-Quality Dataset**: This is the most critical step. You need a dataset consisting of input-output pairs that exemplify the desired behavior. For instruction tuning, this often means `{'instruction': '...', 'response': '...'}` or `{'prompt': '...', 'completion': '...'}`.
3.  **Configure Finetuning Parameters**: Define hyperparameters like learning rate, batch size, number of epochs, and optimization strategy.
4.  **Train the Model**: Feed the dataset to the LLM. During training, the model's weights are updated based on the difference between its predictions and the ground truth outputs in your dataset. Modern SFT often leverages Parameter-Efficient Finetuning (PEFT) techniques like LoRA (Low-Rank Adaptation) or QLoRA (Quantized LoRA) to significantly reduce computational cost and memory footprint, making it accessible even on consumer-grade GPUs.
5.  **Evaluate and Deploy**: After training, evaluate the finetuned model on a held-out test set to ensure it performs as expected. Once satisfied, deploy it for inference.

Hugging Face's `transformers` library, combined with `trl` (Transformer Reinforcement Learning) and `peft` (Parameter-Efficient Finetuning) libraries, provides a robust and user-friendly ecosystem for performing SFT. The `trl` library's `SFTTrainer` specifically streamlines the process of instruction finetuning, handling data formatting, PEFT integration, and training loops efficiently.


In [ ]:
# Install necessary libraries (run this cell if you haven't already)
# !pip install -q transformers datasets peft trl accelerate bitsandbytes

import torch
from datasets import load_dataset
from peft import LoraConfig, AutoPeftModelForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

# 1. Configuration for 4-bit quantization (QLoRA setup)
# This helps reduce memory usage significantly, allowing larger models on smaller GPUs.
# As of 2026, QLoRA is a standard practice for efficient finetuning.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4_bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 2. Load a small, instruction-tuned base model
# We'll use TinyLlama for demonstration due to its small size and fast training.
# For production, you'd typically use larger models like Llama-3-8B, Mistral-7B, or Gemma-2B.
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto", # Automatically maps model layers to available devices
    torch_dtype=torch.bfloat16 # Use bfloat16 for computation with 4-bit quantization
)
model.config.use_cache = False # Disable cache for gradient checkpointing
model.config.pretraining_tp = 1 # Required for some models like Llama

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Set pad token for consistency
tokenizer.padding_side = "right" # Important for decoder-only models

print(f"Model loaded: {model_id}")
print(f"Tokenizer loaded: {model_id}")

# 3. Load a small instruction dataset
# We'll use a subset of the 'tatsu-lab/alpaca' dataset, which is formatted for instruction tuning.
# For real-world applications, you'd use your own domain-specific, high-quality instruction dataset.
dataset_id = "tatsu-lab/alpaca"
dataset = load_dataset(dataset_id, split="train")

# Filter for a smaller subset for faster demonstration
dataset = dataset.shuffle(seed=42).select(range(1000)) # Using 1000 examples for quick run

print(f"Dataset loaded: {dataset_id} with {len(dataset)} examples.")
print("Example data point:")
print(dataset[0])

# 4. Define a formatting function for the dataset
# This function converts the raw dataset entries into a chat-like format suitable for SFT.
# The `trl` SFTTrainer can handle various formats, but this is a common one.
def format_instruction_dataset(example):
    # Alpaca dataset has 'instruction', 'input', and 'output' fields.
    # We combine them into a single prompt string.
    if example["input"]:
        return f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    else:
        return f"### Instruction:\n{example['instruction']}\n### Response:\n{example['output']}"

# 5. Configure LoRA (Parameter-Efficient Finetuning)
# LoRA allows finetuning only a small number of new parameters, making it much more efficient.
# This is critical for finetuning large models on limited hardware.
lora_config = LoraConfig(
    r=16, # LoRA attention dimension
    lora_alpha=32, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Modules to apply LoRA to
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Only finetune bias weights if specified
    task_type="CAUSAL_LM", # Task type for causal language modeling
)

# 6. Define Training Arguments
# These arguments control the training process, including learning rate, batch size, etc.
training_arguments = TrainingArguments(
    output_dir="./sft_results", # Directory to save checkpoints and logs
    num_train_epochs=1, # Number of training epochs (keep low for demo)
    per_device_train_batch_size=2, # Batch size per GPU
    gradient_accumulation_steps=4, # Accumulate gradients over multiple steps
    optim="paged_adamw_8bit", # Optimizer (paged_adamw_8bit is memory efficient)
    learning_rate=2e-4, # Learning rate
    fp16=False, # Use bfloat16 instead of fp16 for better stability with 4-bit quantization
    bf16=True, # Enable bfloat16 training
    max_grad_norm=0.3, # Max gradient norm for clipping
    warmup_ratio=0.03, # Warmup ratio for learning rate scheduler
    lr_scheduler_type="constant", # Learning rate scheduler type
    logging_steps=25, # Log training metrics every N steps
    save_strategy="epoch", # Save model checkpoint every epoch
    report_to="none", # Disable reporting to external services like Weights & Biases for simplicity
    push_to_hub=False, # Don't push to Hugging Face Hub for demo
)

# 7. Initialize SFTTrainer
# The SFTTrainer from `trl` simplifies the SFT process, integrating PEFT and data formatting.
sft_trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    dataset_text_field=None, # We'll use `formatting_func` instead of a single text field
    formatting_func=format_instruction_dataset, # Custom function to format dataset examples
    max_seq_length=512, # Maximum sequence length for training
    tokenizer=tokenizer,
    args=training_arguments,
    packing=True, # Pack multiple short examples into one sequence for efficiency
)

# 8. Train the model
print("Starting SFT training...")
sft_trainer.train()
print("SFT training complete!")

# 9. Save the finetuned adapter
# Only the LoRA adapters are saved, not the full model, which is very space-efficient.
sft_trainer.model.save_pretrained("./finetuned_tinyllama_adapter")
tokenizer.save_pretrained("./finetuned_tinyllama_adapter")
print("Finetuned adapter saved to ./finetuned_tinyllama_adapter")

# 10. Load the finetuned model and test inference
# To perform inference, we load the base model and then merge the LoRA adapters.
print("\n--- Testing Finetuned Model ---")

# Clear GPU memory before loading for inference if needed
del model
del sft_trainer
torch.cuda.empty_cache()

# Load the base model again (or use the original if not deleted)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Load the finetuned adapter and merge it with the base model
finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    "./finetuned_tinyllama_adapter",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
finetuned_model = finetuned_model.merge_and_unload() # Merge LoRA weights into the base model

finetuned_tokenizer = AutoTokenizer.from_pretrained("./finetuned_tinyllama_adapter")

def generate_response(prompt, model, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:\n")[-1].strip()

# Example prompt for testing
test_prompt = "### Instruction:\nExplain the concept of quantum entanglement in simple terms.\n### Response:\n"
print(f"Prompt: {test_prompt}")
print(f"Finetuned Response: {generate_response(test_prompt, finetuned_model, finetuned_tokenizer)}")

test_prompt_2 = "### Instruction:\nWrite a short, encouraging message for someone starting a new job.\n### Response:\n"
print(f"Prompt: {test_prompt_2}")
print(f"Finetuned Response: {generate_response(test_prompt_2, finetuned_model, finetuned_tokenizer)}")


### Interpreting the Code Output and Performance Trade-offs

After running the SFT code, you'll observe training logs that typically show the loss decreasing over steps. A decreasing loss indicates that the model is learning and adjusting its weights to better predict the desired outputs from your training data. For a short demo with a small dataset, the loss might not drop dramatically, but in a full training run, you'd expect a significant reduction.

**Key Observations from Training Logs:**

*   **Loss**: The primary metric. A steady decrease is good. If it plateaus or increases, it might indicate issues like a too-high learning rate, insufficient data, or overfitting.
*   **Learning Rate**: You'll see how the learning rate scheduler adjusts the learning rate over time (e.g., warmup, then constant or decay).
*   **Epochs/Steps**: The progress through your dataset.

**Performance Trade-offs and Considerations:**

1.  **Data Quality vs. Quantity**: This is paramount. A smaller, high-quality, diverse, and well-formatted dataset will almost always yield better results than a massive, noisy, or poorly formatted one. Garbage in, garbage out applies strongly to SFT. Ensure your data covers the full range of desired behaviors and edge cases.

2.  **Model Size**: Larger base models (e.g., Llama-3-70B) generally have superior reasoning and generalization capabilities compared to smaller ones (e.g., TinyLlama-1.1B). However, they require significantly more computational resources (GPU memory, VRAM, training time). The choice depends on your budget, hardware, and performance requirements.

3.  **Parameter-Efficient Finetuning (PEFT) - LoRA/QLoRA**: As demonstrated, PEFT techniques like LoRA are game-changers. They allow finetuning massive models (even 70B parameters) on consumer-grade GPUs (e.g., 24GB VRAM) by only training a small fraction of new parameters (the 'adapters').
    *   **Pros**: Dramatically reduced memory footprint, faster training, smaller checkpoint sizes (only adapters are saved), and less prone to catastrophic forgetting of the base model's knowledge.
    *   **Cons**: May sometimes lead to slightly lower performance compared to full finetuning (though often negligible for many tasks), and merging adapters can be slow for inference if not done beforehand.

4.  **Hyperparameters**: Tuning `learning_rate`, `batch_size`, `gradient_accumulation_steps`, `num_train_epochs`, and `max_seq_length` is crucial. These values are highly dependent on your specific model, dataset, and task. Experimentation is key.

5.  **Hardware**: SFT, even with QLoRA, is compute-intensive. Access to powerful GPUs (e.g., NVIDIA A100s, H100s, or even consumer RTX 4090s) is essential. Cloud platforms like Google Cloud, AWS, or Azure provide scalable GPU instances.

6.  **Overfitting**: If you train for too many epochs on a small dataset, the model might overfit, meaning it performs exceptionally well on the training data but poorly on unseen data. Monitoring a validation set (if available) is crucial to detect this.

**Typical Use Cases for SFT:**

*   **Custom Chatbots**: Training a general LLM to become an expert customer service agent for a specific product or service.
*   **Domain-Specific Content Generation**: Generating legal documents, medical reports, financial analyses, or technical documentation that requires specialized knowledge and terminology.
*   **Code Generation/Refinement**: Adapting an LLM to generate code in a specific programming language, framework, or internal coding style.
*   **Creative Writing**: Guiding an LLM to write poetry, stories, or marketing copy in a particular author's style or tone.
*   **Data Augmentation**: Creating synthetic data that adheres to specific patterns or formats for further model training.

By mastering SFT, ML engineers can unlock the full potential of pre-trained LLMs, transforming them into highly specialized and performant agents for a vast array of real-world applications.


### Resources

*   **Hugging Face Transformers Documentation**: The primary resource for working with LLMs and finetuning. [https://huggingface.co/docs/transformers/index](https://huggingface.co/docs/transformers/index)
*   **Hugging Face `trl` Library**: Specifically for SFT and RLHF. The `SFTTrainer` is a key component. [https://huggingface.co/docs/trl/main/en/sft_trainer](https://huggingface.co/docs/trl/main/en/sft_trainer)
*   **Hugging Face `peft` Library**: Learn more about LoRA and other parameter-efficient finetuning methods. [https://huggingface.co/docs/peft/main/en/index](https://huggingface.co/docs/peft/main/en/index)
*   **QLoRA: Efficient Finetuning of Quantized LLMs**: Original paper introducing QLoRA. [https://arxiv.org/abs/2305.14314](https://arxiv.org/abs/2305.14314)
*   **LoRA: Low-Rank Adaptation of Large Language Models**: Original paper introducing LoRA. [https://arxiv.org/abs/2106.09685](https://arxiv.org/abs/2106.09685)
*   **Hugging Face Blog Post on SFT**: A practical guide to finetuning LLMs. [https://huggingface.co/blog/sft_trainer](https://huggingface.co/blog/sft_trainer)
*   **Google AI Studio / Gemini API Documentation**: While this lesson focuses on open-source models, Google AI Studio offers a platform for interacting with and finetuning Google's proprietary models. [https://ai.google.dev/](https://ai.google.dev/)
